# Exploratory Data Analysis: Dogs vs Cats Dataset

This notebook explores the raw dataset to understand its properties before training.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

RAW_DIR = Path("../data/raw")
all_images = sorted(RAW_DIR.glob("*.jpg"))
print(f"Total images found: {len(all_images)}")

## 1. Class Distribution

In [ ]:
cat_images = [p for p in all_images if p.name.startswith("cat.")]
dog_images = [p for p in all_images if p.name.startswith("dog.")]

print(f"Cats: {len(cat_images)}")
print(f"Dogs: {len(dog_images)}")
print(f"Ratio: {len(cat_images) / len(dog_images):.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Cat", "Dog"], [len(cat_images), len(dog_images)], color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Count")
ax.set_title("Class Distribution")
for i, v in enumerate([len(cat_images), len(dog_images)]):
    ax.text(i, v + 100, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Image Size Distribution

In [ ]:
widths = []
heights = []
corrupted = []

for img_path in all_images:
    try:
        with Image.open(img_path) as img:
            img.verify()
        with Image.open(img_path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except Exception as e:
        corrupted.append((img_path.name, str(e)))

widths = np.array(widths)
heights = np.array(heights)

print(f"Successfully read: {len(widths)} images")
print(f"Corrupted: {len(corrupted)} images")
print(f"\nWidth  — min: {widths.min()}, max: {widths.max()}, mean: {widths.mean():.0f}, median: {np.median(widths):.0f}")
print(f"Height — min: {heights.min()}, max: {heights.max()}, mean: {heights.mean():.0f}, median: {np.median(heights):.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths, bins=50, color="#4C72B0", edgecolor="white")
axes[0].set_title("Width Distribution")
axes[0].set_xlabel("Pixels")

axes[1].hist(heights, bins=50, color="#DD8452", edgecolor="white")
axes[1].set_title("Height Distribution")
axes[1].set_xlabel("Pixels")

axes[2].scatter(widths, heights, alpha=0.05, s=3, color="#55A868")
axes[2].set_title("Width vs Height")
axes[2].set_xlabel("Width")
axes[2].set_ylabel("Height")

plt.tight_layout()
plt.show()

## 3. Sample Images Grid

16 random cats and 16 random dogs.

In [ ]:
rng = np.random.default_rng(42)

sample_cats = rng.choice(cat_images, 16, replace=False)
sample_dogs = rng.choice(dog_images, 16, replace=False)

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
fig.suptitle("Random Samples: Cats (left) vs Dogs (right)", fontsize=14)

for i in range(4):
    for j in range(4):
        # Cats on the left
        img = Image.open(sample_cats[i * 4 + j]).convert("RGB")
        axes[i][j].imshow(img)
        axes[i][j].axis("off")
        if i == 0:
            axes[i][j].set_title("Cat", fontsize=10)

        # Dogs on the right
        img = Image.open(sample_dogs[i * 4 + j]).convert("RGB")
        axes[i][j + 4].imshow(img)
        axes[i][j + 4].axis("off")
        if i == 0:
            axes[i][j + 4].set_title("Dog", fontsize=10)

plt.tight_layout()
plt.show()

## 4. Corrupted / Unreadable Images

In [ ]:
if corrupted:
    print(f"Found {len(corrupted)} corrupted images:\n")
    for name, error in corrupted:
        print(f"  {name}: {error}")
else:
    print("No corrupted images found!")

## 5. File Format Distribution

In [ ]:
from collections import Counter

formats = []
modes = []
for img_path in all_images:
    try:
        with Image.open(img_path) as img:
            formats.append(img.format)
            modes.append(img.mode)
    except Exception:
        pass

print("File formats:")
for fmt, count in Counter(formats).most_common():
    print(f"  {fmt}: {count}")

print(f"\nColor modes:")
for mode, count in Counter(modes).most_common():
    print(f"  {mode}: {count}")

## 6. File Size Distribution

In [ ]:
file_sizes = np.array([p.stat().st_size / 1024 for p in all_images])  # in KB

print(f"File size (KB) — min: {file_sizes.min():.1f}, max: {file_sizes.max():.1f}, mean: {file_sizes.mean():.1f}, median: {np.median(file_sizes):.1f}")
print(f"Total dataset size: {file_sizes.sum() / 1024:.0f} MB")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(file_sizes, bins=50, color="#4C72B0", edgecolor="white")
ax.set_title("File Size Distribution")
ax.set_xlabel("Size (KB)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Summary

Key findings from EDA:
- **Class balance:** Check if cats and dogs are roughly 50/50
- **Image sizes:** Vary widely — need to resize to a fixed size (224x224) for training
- **Corrupted images:** Any found should be removed in the preprocessing step
- **Color modes:** Some images may be grayscale or RGBA — need to convert to RGB during preprocessing